<a href="https://colab.research.google.com/github/sarwathasan72-svg/Masai_IIT_AIML/blob/main/evaluating_regression_performance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluating Regression Performance
### Hands-on Notebook — MSE & R-squared

**Learning Objectives**
1. Calculate and interpret **Mean Squared Error (MSE)**
2. Analyze **R-squared** for model fit

**Subtopics:** Regression Metrics · Mean Squared Error (MSE) · R-squared

---
**The situation:** you're predicting Bengaluru apartment prices (₹ Lakhs) from size (sqft). You've already fit a regression line. Now you need to answer, with numbers — not a glance at a scatter plot — *how good is this model?*


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

plt.rcParams['figure.figsize'] = (7, 4.5)
np.random.seed(42)


## 1. The dataset

We'll use the same 5-apartment example from the slides, plus a slightly larger synthetic dataset so the metrics feel less like a toy.


In [ ]:
# The exact worked example from the slide deck
demo = pd.DataFrame({
    "size_sqft": [600, 800, 1000, 1200, 1500],
    "actual_price_L": [40, 55, 58, 65, 90],
    "predicted_price_L": [40, 50, 60, 70, 85],
})
demo


## 2. Mean Squared Error — from scratch

Recall the formula:

$$MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

Let's build it step by step, matching the table on the slides.


In [ ]:
demo["residual"] = demo["actual_price_L"] - demo["predicted_price_L"]
demo["squared_residual"] = demo["residual"] ** 2
demo


In [ ]:
n = len(demo)
sum_squared_residuals = demo["squared_residual"].sum()
mse_manual = sum_squared_residuals / n

print(f"Sum of squared residuals: {sum_squared_residuals}")
print(f"n: {n}")
print(f"MSE (manual) = {sum_squared_residuals} / {n} = {mse_manual}")


**Checkpoint:** this should match the slide — `MSE = 79 / 5 = 15.8`.

Now verify with scikit-learn's built-in function — this is what you'll use in practice.


In [ ]:
mse_sklearn = mean_squared_error(demo["actual_price_L"], demo["predicted_price_L"])
print(f"MSE (sklearn) = {mse_sklearn}")
assert np.isclose(mse_manual, mse_sklearn), "Manual and sklearn MSE should match!"
print("Manual calculation matches sklearn ✔")


### RMSE — putting MSE back into real units

MSE is in ₹Lakhs², which is awkward to reason about. Taking the square root brings it back to ₹Lakhs.


In [ ]:
rmse_manual = np.sqrt(mse_manual)
print(f"RMSE = √{mse_manual} = {rmse_manual:.2f} ₹ Lakhs")
print("Interpretation: our predictions are typically off by roughly this much, in real terms.")


### Visualizing residuals

The dashed lines below are exactly what gets squared and averaged in MSE.


In [ ]:
fig, ax = plt.subplots()
ax.scatter(demo["size_sqft"], demo["actual_price_L"], color="#F3F7FB", edgecolor="#122238", s=70, zorder=3, label="Actual")
ax.plot(demo["size_sqft"], demo["predicted_price_L"], color="#F2A93B", linewidth=2, marker="s", markersize=5, label="Predicted (model line)")

for _, row in demo.iterrows():
    ax.plot([row["size_sqft"], row["size_sqft"]],
            [row["actual_price_L"], row["predicted_price_L"]],
            color="#E15B64", linestyle="--", linewidth=1.5, zorder=2)

ax.set_facecolor("#12223822")
ax.set_xlabel("Size (sqft)")
ax.set_ylabel("Price (₹ Lakhs)")
ax.set_title("Residuals: the gap MSE squares and averages")
ax.legend()
plt.tight_layout()
plt.show()


## 3. R-squared — from scratch

R² compares your model's error to the error of the simplest possible baseline: **always predict the mean**.

$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}} \qquad
SS_{res} = \sum (y_i - \hat{y}_i)^2 \qquad
SS_{tot} = \sum (y_i - \bar{y})^2$$


In [ ]:
y_actual = demo["actual_price_L"].values
y_pred = demo["predicted_price_L"].values

y_mean = y_actual.mean()
ss_res = np.sum((y_actual - y_pred) ** 2)
ss_tot = np.sum((y_actual - y_mean) ** 2)

r2_manual = 1 - (ss_res / ss_tot)

print(f"Mean actual price (ȳ): {y_mean}")
print(f"SS_res (model's error):    {ss_res}")
print(f"SS_tot (baseline's error): {ss_tot:.2f}")
print(f"R² = 1 - {ss_res}/{ss_tot:.2f} = {r2_manual:.4f}")


In [ ]:
r2_sklearn = r2_score(y_actual, y_pred)
print(f"R² (sklearn) = {r2_sklearn:.4f}")
assert np.isclose(r2_manual, r2_sklearn), "Manual and sklearn R² should match!"
print("Manual calculation matches sklearn ✔")


**Interpretation:** an R² of ~0.94 means the model explains about 94% of the variation in apartment prices — a strong fit. The remaining ~6% is unexplained (noise, or factors the model doesn't capture, like floor number or locality).


### Seeing R² fail: comparing against a bad model

To make the "compared to what?" idea concrete, let's compute R² for a deliberately bad baseline — always predicting the mean price — and a genuinely poor model.


In [ ]:
# Baseline: always predict the mean
baseline_pred = np.full_like(y_actual, fill_value=y_mean, dtype=float)
r2_baseline = r2_score(y_actual, baseline_pred)
print(f"R² of the 'always predict the mean' baseline: {r2_baseline:.4f}  (by definition, always 0)")

# A genuinely bad model: predictions roughly reversed / noisy
bad_pred = np.array([90, 40, 85, 55, 58])  # scrambled on purpose
r2_bad = r2_score(y_actual, bad_pred)
print(f"R² of a scrambled/poor model:                 {r2_bad:.4f}  (can go negative!)")


Notice: a negative R² means the model is doing **worse** than simply guessing the average every time. This is a real, useful signal — MSE alone wouldn't tell you that so clearly.


## 4. A larger, more realistic dataset

Let's scale up: generate a synthetic dataset of 40 apartments, fit a real `LinearRegression` model with scikit-learn, and evaluate it properly (train/test split included).


In [ ]:
n_samples = 40
size = np.random.normal(1000, 300, n_samples).clip(400, 2000)
noise = np.random.normal(0, 8, n_samples)
price = 0.05 * size + 10 + noise   # true underlying relationship + noise

df = pd.DataFrame({"size_sqft": size, "price_L": price})
df.head()


In [ ]:
from sklearn.model_selection import train_test_split

X = df[["size_sqft"]]
y = df["price_L"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred_test = model.predict(X_test)

test_mse = mean_squared_error(y_test, y_pred_test)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, y_pred_test)

print(f"Test MSE:  {test_mse:.2f}")
print(f"Test RMSE: {test_rmse:.2f} ₹ Lakhs")
print(f"Test R²:   {test_r2:.4f}")


In [ ]:
fig, ax = plt.subplots()
ax.scatter(X_train, y_train, color="#5F7996", alpha=0.6, s=40, label="Train")
ax.scatter(X_test, y_test, color="#F3F7FB", edgecolor="#122238", s=55, zorder=3, label="Test (actual)")

x_line = np.linspace(df["size_sqft"].min(), df["size_sqft"].max(), 100).reshape(-1, 1)
ax.plot(x_line, model.predict(x_line), color="#F2A93B", linewidth=2, label="Fitted line")

ax.set_xlabel("Size (sqft)")
ax.set_ylabel("Price (₹ Lakhs)")
ax.set_title(f"Fitted model  |  Test RMSE = {test_rmse:.2f}   Test R² = {test_r2:.3f}")
ax.legend()
plt.tight_layout()
plt.show()


## 5. Exercises

Work through these before checking the solutions. Use the `df` dataset above unless told otherwise.


### Exercise 1 — By hand, then verify
A model made these predictions on 4 new apartments:

| Actual | Predicted |
|---|---|
| 48 | 45 |
| 62 | 68 |
| 71 | 70 |
| 39 | 50 |

Compute the MSE **manually** (residual → square → average), then verify with `mean_squared_error`.


In [ ]:
# TODO: your code here
actual_ex1 = np.array([48, 62, 71, 39])
predicted_ex1 = np.array([45, 68, 70, 50])

# 1. compute residuals
# 2. square them
# 3. average them
# 4. verify against mean_squared_error(...)


<details><summary>Show solution</summary>

```python
residuals = actual_ex1 - predicted_ex1
squared = residuals ** 2
mse_ex1 = squared.mean()
print(mse_ex1)                                   # 46.5
print(mean_squared_error(actual_ex1, predicted_ex1))  # 46.5 — matches
```
</details>


### Exercise 2 — Interpret, don't just calculate
Using the same 4 predictions above, compute R². Then answer in a markdown cell:
- Is this model doing better or worse than always predicting the mean?
- Would you deploy this model? Why or why not?


In [ ]:
# TODO: compute R2 for actual_ex1 / predicted_ex1



<details><summary>Show solution</summary>

```python
r2_ex1 = r2_score(actual_ex1, predicted_ex1)
print(r2_ex1)   # roughly 0.29
```
An R² around 0.29 means the model explains only ~29% of the variance — better than the mean baseline (R²=0), but weak. Whether to deploy depends on the domain: 0.29 might be unacceptable for pricing decisions but could be reasonable for a first noisy pass.
</details>


### Exercise 3 — Outlier sensitivity
Take the 5-apartment `demo` dataset. Change the **last** predicted price from 85 to 40 (simulating one very bad prediction), and recompute MSE and R².

- How much did MSE change? How much did R² change?
- What does this tell you about relying on a single metric?


In [ ]:
# TODO: copy demo, corrupt the last prediction, recompute MSE and R2



<details><summary>Show solution</summary>

```python
corrupted_pred = demo["predicted_price_L"].copy()
corrupted_pred.iloc[-1] = 40

print("New MSE:", mean_squared_error(demo["actual_price_L"], corrupted_pred))
print("New R2:", r2_score(demo["actual_price_L"], corrupted_pred))
```
Both metrics get noticeably worse from a single bad point — MSE jumps because the error is squared, and R² drops because SS_res grows. This is exactly why outliers deserve attention before trusting either metric at face value.
</details>


## 6. Recap

- **MSE** = average squared residual. Lower is better; same-scale comparisons only; sensitive to outliers; report `RMSE = √MSE` for interpretable units.
- **R²** = 1 − (model's error ÷ baseline's error), where the baseline always predicts the mean. Ranges up to 1; can go negative; unitless, so easier to compare across contexts than MSE.
- Report **both** together — MSE/RMSE for magnitude of error, R² for relative explanatory power.

**Next up:** other regression metrics (MAE, Adjusted R²) and how metric choice changes with outliers and business context.
